In [3]:
import scipy.io

# Ruta relativa al archivo
mat_path = "/home/alumnos/mburgosc/tfg/egm_reconstruction/Data/TwoRotors_181219/EGMs.mat"

mat_data = scipy.io.loadmat(mat_path)
x = mat_data['x']

print(f"Forma de x: {x.shape}")
print(f"Primeros valores:\n{x[:5, :5]}")


Forma de x: (2048, 2499)
Primeros valores:
[[-0.58787677 -0.64342173 -0.69716551 -0.74769737 -0.79379689]
 [-0.33220731 -0.39333809 -0.45269263 -0.50907527 -0.56145873]
 [-0.68672067 -0.74213266 -0.79568582 -0.84587382 -0.89139189]
 [-0.31360689 -0.37635283 -0.43731063 -0.49529788 -0.54929688]
 [-0.60280564 -0.66013307 -0.715649   -0.76793588 -0.8157605 ]]


In [4]:
for key in mat_data:
    if not key.startswith("__"):  # ignora __header__, __globals__, etc.
        print(f"{key}: {type(mat_data[key])}, shape: {mat_data[key].shape}")

x: <class 'numpy.ndarray'>, shape: (2048, 2499)


In [8]:
from scipy.io import whosmat

mat_files = ["BSPs.mat", "driver_position.mat", "driver_position_2.mat"]

for file in mat_files:
    print(f"\n{file}:")
    vars = whosmat(f"../../Data/TwoRotors_181219/{file}")
    for var in vars:
        print(f"  {var[0]} - shape: {var[1]}, type: {var[2]}")


BSPs.mat:
  y - shape: (659, 2499), type: double

driver_position.mat:
  driver_position - shape: (2499, 2), type: cell

driver_position_2.mat:
  driver_position - shape: (2499, 2), type: cell


In [9]:
from scipy.io import loadmat

mat = loadmat("../../Data/TwoRotors_181219/driver_position.mat", squeeze_me=True)
driver_position = mat['driver_position']  # shape: (2499, 2)

In [14]:
print(type(driver_position[0, 0]))
print(driver_position[0, 0])  # o [0][0]
print(driver_position)

<class 'int'>
0
[[0 array([], dtype=uint8)]
 [0 array([], dtype=uint8)]
 [0 array([], dtype=uint8)]
 ...
 [0 array([], dtype=uint8)]
 [0 array([], dtype=uint8)]
 [0 array([], dtype=uint8)]]


In [16]:
import numpy as np

fs = 500  # Hz, confirma esto

dominant_freqs = []
for i in range(x.shape[1]):  # para cada canal
    freqs = np.fft.rfftfreq(x.shape[0], d=1/fs)
    fft_vals = np.abs(np.fft.rfft(x[:, i]))
    df = freqs[np.argmax(fft_vals)]
    dominant_freqs.append(df)

dominant_freqs = np.array(dominant_freqs)

top_indices = np.argsort(dominant_freqs)[-2:]
print(f"Canales con DF más alta: {top_indices}")
print(f"DFs: {dominant_freqs[top_indices]}")


Canales con DF más alta: [ 273 2479]
DFs: [1.70898438 1.953125  ]


In [42]:
import scipy.io

# Ruta relativa al archivo
mat_path = "/home/alumnos/mburgosc/tfg/egm_reconstruction/Data/TwoRotors_181219/driver_position.mat"

mat_data = scipy.io.loadmat(mat_path)

print(mat_data.keys())
print(mat_data['driver_position'])


dict_keys(['__header__', '__version__', '__globals__', 'driver_position'])
[[array([[0]], dtype=uint8) array([], shape=(0, 0), dtype=uint8)]
 [array([[0]], dtype=uint8) array([], shape=(0, 0), dtype=uint8)]
 [array([[0]], dtype=uint8) array([], shape=(0, 0), dtype=uint8)]
 ...
 [array([[0]], dtype=uint8) array([], shape=(0, 0), dtype=uint8)]
 [array([[0]], dtype=uint8) array([], shape=(0, 0), dtype=uint8)]
 [array([[0]], dtype=uint8) array([], shape=(0, 0), dtype=uint8)]]


In [53]:
mat_path = "/home/alumnos/mburgosc/tfg/egm_reconstruction/Data/regions.mat"
mat_data = scipy.io.loadmat(mat_path)

print(mat_data.keys())
print((mat_data))

dict_keys(['__header__', '__version__', '__globals__', 'regions'])
{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: s\xe1. abr. 11 20:19:14 2020', '__version__': '1.0', '__globals__': [], 'regions': array([[5, 5, 5, ..., 2, 2, 2]], dtype=uint8)}


In [ ]:
import numpy as np
import os
from scipy.io import loadmat

def get_multilabel(model_name, n_regiones=6):
    path_driver_position = f"../../DATA_USE/{model_name}/driver_position.mat"

    if not os.path.exists(path_driver_position):
        return np.zeros(n_regiones, dtype=int)  # Sinusal

    driver_position = loadmat(path_driver_position)["driver_position"]
    nodes_aux = driver_position[:, 1]
    driver_position = driver_position[:, 0]

    regions = loadmat("../../DATA_USE/regions.mat")["regions"][0]

    labels = []
    for idx in range(len(driver_position)):
        if driver_position[idx] != 0:
            node_idx = int(nodes_aux[idx][0, 0]) - 1  # índice en regiones
            if 0 <= node_idx < len(regions):
                region_label = int(regions[node_idx])
                labels.append(region_label)

    # Convertimos a binario multi-etiqueta
    multilabel = np.zeros(n_regiones, dtype=int)
    for r in labels:
        if 1 <= r <= n_regiones:
            multilabel[r - 1] = 1

    return multilabel



labels_multilabel = [get_multilabel(name) for name in modelos_unicos]
print(labels_multilabel)